# FinSure AI: Complete Training Pipeline (SFT + GRPO/ART + DSPy)

This notebook provides a **modular, production-ready** training pipeline for fine-tuning Qwen3-4B on financial QA data.

## 🏗️ Architecture

**Phase 1: Supervised Fine-Tuning (SFT)**
- LoRA-based training on financial QA dataset
- 4-bit quantization for memory efficiency
- Merges adapters into base model

**Phase 2: GRPO/ART Reinforcement Learning**
- Group Relative Policy Optimization
- Custom reward function with semantic similarity
- Penalties for verbosity, repetition, and hallucinations

**Phase 3: DSPy Inference Testing**
- Chain-of-Thought reasoning
- Validation on test examples
- Performance metrics

## 🚀 Usage

**Run full pipeline:**
```bash
modal run modal_finetune.ipynb
```

**Run individual phases:**
```bash
modal run modal_finetune.ipynb::sft_training
modal run modal_finetune.ipynb::grpo_training
modal run modal_finetune.ipynb::dspy_inference_test
```

**Download trained models:**
```bash
modal run modal_finetune.ipynb::download_models
```

## 📋 Requirements
- Modal account (`modal token new`)
- GPU: A10G (24GB VRAM)
- ~2-3 hours for full pipeline

## 💾 Output
Models saved to Modal Volume `finsure-models`:
- `/models/finetuned_qwen` - LoRA adapters
- `/models/merged_finetuned_qwen` - Merged SFT model  
- `/models/qwen-4b-art` - Final GRPO/ART model

In [ ]:
import modal
from modal import Image, App, gpu

app = App("finsure-modal-notebook")

# Requirements embedded in the notebook
requirements_list = [
    "torch>=2.0.0",
    "transformers>=4.40.0",
    "datasets>=2.18.0",
    "peft>=0.10.0",
    "trl>=0.8.0",
    "accelerate>=0.28.0",
    "dspy-ai>=2.4.0",
    "modal>=0.60.0",
    "jupyter>=1.0.0",
    "ipykernel>=6.0.0",
    "sentence-transformers",
    "evaluate>=0.4.0",
    "tqdm>=4.0.0",
]

# Create image with all dependencies
image = (
    Image.debian_slim()
    .pip_install(*requirements_list)
    .run_commands(
        "apt-get update && apt-get install -y git",
        "pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121",
        "pip install bitsandbytes",
        "pip install git+https://github.com/openpipe/art.git"
    )
)

volume = modal.Volume.from_name("finsure-models", create_if_missing=True)

In [ ]:
# ============================================================================
# REWARD FUNCTION FOR GRPO TRAINING
# ============================================================================

REWARD_CODE = '''"""
Reward function for GRPO (Group Relative Policy Optimization) training.
Uses sentence-transformers for semantic similarity and heuristic checks.
"""

from sentence_transformers import SentenceTransformer
import re

# Initialize sentence transformer model (lazy loading)
_model = None

def _get_model():
    """Lazy load the sentence transformer model."""
    global _model
    if _model is None:
        _model = SentenceTransformer('all-MiniLM-L6-v2')
    return _model

def compute_reward(prompt, generated, ground_truth):
    """
    Compute reward for a generated answer given a prompt and ground truth.
    
    Args:
        prompt: The input prompt/question
        generated: The generated answer from the model
        ground_truth: The ground truth answer
    
    Returns:
        float: Reward score (typically between -1 and 1)
    """
    if not generated or not ground_truth:
        return -0.5
    
    model = _get_model()
    
    # Base similarity score using sentence transformers
    embeddings = model.encode([generated, ground_truth], convert_to_tensor=True)
    similarity = float((embeddings[0] @ embeddings[1]) / (embeddings[0].norm() * embeddings[1].norm()))
    
    reward = similarity
    
    # Extract key tokens from ground truth (non-stop words)
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 
                  'of', 'with', 'by', 'from', 'is', 'are', 'was', 'were', 'be', 'been'}
    gt_words = set(re.findall(r'\\b\\w+\\b', ground_truth.lower())) - stop_words
    gen_words = set(re.findall(r'\\b\\w+\\b', generated.lower()))
    
    # Bonus: key tokens from ground truth appear in generated
    key_matches = len(gt_words & gen_words)
    if key_matches > 0:
        reward += 0.1 * min(key_matches / max(len(gt_words), 1), 1.0)
    
    # Penalty: generated answer is much longer than ground truth
    len_ratio = len(generated) / max(len(ground_truth), 1)
    if len_ratio > 2.0:  # Generated is more than 2x longer
        reward -= 0.2
    
    # Penalty: repetition detection (simple heuristic)
    sentences = re.split(r'[.!?]+', generated)
    if len(sentences) > 1:
        # Check for repeated sentences
        unique_sentences = set(s.strip().lower() for s in sentences if len(s.strip()) > 10)
        if len(unique_sentences) < len(sentences) * 0.7:  # More than 30% repetition
            reward -= 0.2
    
    # Penalty: off-topic or hallucinated facts (simple heuristic)
    # Check if generated answer contains common hallucination indicators
    hallucination_indicators = [
        'i do not have access',
        'i cannot provide',
        'i am unable to',
        'as an ai',
        'i apologize',
    ]
    generated_lower = generated.lower()
    if any(indicator in generated_lower for indicator in hallucination_indicators):
        reward -= 0.2
    
    # Ensure reward is bounded
    reward = max(-1.0, min(1.0, reward))
    
    return float(reward)'''


In [ ]:
# ============================================================================
# PHASE 1: SUPERVISED FINE-TUNING (SFT)
# ============================================================================

@app.function(
    image=image,
    gpu="A10G",
    volumes={"/models": volume},
    timeout=3600
)
def sft_training():
    """
    Phase 1: Supervised Fine-Tuning with LoRA on financial QA dataset.
    
    Returns:
        str: Path to the merged model
    """
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
    from datasets import load_dataset
    from peft import LoraConfig, prepare_model_for_kbit_training, AutoPeftModelForCausalLM
    from trl import SFTTrainer
    
    try:
        print("\n" + "="*80)
        print("🚀 PHASE 1: SUPERVISED FINE-TUNING (SFT)")
        print("="*80 + "\n")
        
        # Model setup
        model_name = "Qwen/Qwen3-4B-Instruct-2507"
        print(f"📥 Loading tokenizer: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # GPU check
        assert torch.cuda.is_available(), "GPU not available!"
        print(f"✅ GPU: {torch.cuda.get_device_name()}\n")

        # Load dataset
        print("📥 Loading dataset: virattt/financial-qa-10K")
        dataset = load_dataset("virattt/financial-qa-10K", split="train")
        print(f"✅ Dataset loaded: {len(dataset)} examples")

        # Format function for chat template
        def format_example(example):
            messages = [
                {"role": "system", "content": "You are a financial expert. Provide a concise answer to the question based on the given context."},
                {"role": "user", "content": f"Context: {example['context']}\n\nQuestion: {example['question']}"},
                {"role": "assistant", "content": example["answer"]}
            ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            return {"text": text}

        # Apply formatting (subsample to 1000 for testing; remove .select() for full training)
        print("🔄 Formatting dataset...")
        dataset = dataset.shuffle(seed=42).select(range(1000)).map(format_example)
        print(f"✅ Formatted {len(dataset)} examples\n")

        # Quantization config
        print("⚙️  Configuring 4-bit quantization...")
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )

        # Load model
        print(f"📥 Loading model: {model_name} (this may take a few minutes)...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True
        )
        print("✅ Model loaded\n")

        # LoRA config
        print("🔧 Configuring LoRA...")
        peft_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "v_proj"]
        )

        model = prepare_model_for_kbit_training(model)
        print("✅ LoRA configured\n")

        # Training arguments
        print("⚙️  Setting up training arguments...")
        args = TrainingArguments(
            output_dir="/models/finetuned_qwen",
            num_train_epochs=1,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            fp16=True,
            save_steps=500,
            logging_steps=100,
            optim="paged_adamw_8bit",
            weight_decay=0.01,
            warmup_steps=100,
            evaluation_strategy="no",
            report_to="none"
        )

        # Trainer
        print("🔧 Initializing SFTTrainer...")
        trainer = SFTTrainer(
            model=model,
            train_dataset=dataset,
            peft_config=peft_config,
            dataset_text_field="text",
            tokenizer=tokenizer,
            args=args,
            max_seq_length=2048
        )
        print("✅ Trainer initialized\n")

        # Train
        print("🚀 Starting training...")
        print("-" * 80)
        trainer.train()
        print("-" * 80)
        print("✅ Training complete!\n")

        # Save LoRA adapters
        print("💾 Saving LoRA adapters...")
        trainer.save_model("/models/finetuned_qwen")
        print("✅ LoRA adapters saved\n")

        # Load and merge
        print("🔄 Loading PEFT model for merging...")
        merged_model = AutoPeftModelForCausalLM.from_pretrained(
            "/models/finetuned_qwen",
            device_map="auto",
            torch_dtype=torch.float16
        )
        
        print("🔄 Merging LoRA adapters into base model...")
        merged_model = merged_model.merge_and_unload()

        # Save merged model
        print("💾 Saving merged model...")
        merged_model.save_pretrained("/models/merged_finetuned_qwen")
        tokenizer.save_pretrained("/models/merged_finetuned_qwen")
        
        volume.commit()
        
        print("\n" + "="*80)
        print("✅ PHASE 1 COMPLETE: SFT Model Saved")
        print("📍 Location: /models/merged_finetuned_qwen")
        print("="*80 + "\n")
        
        return "/models/merged_finetuned_qwen"
        
    except Exception as e:
        print(f"\n❌ ERROR in SFT Training: {str(e)}")
        raise


In [ ]:
# ============================================================================
# PHASE 2: GRPO/ART REINFORCEMENT LEARNING
# ============================================================================

@app.function(
    image=image,
    gpu="A10G",
    volumes={"/models": volume},
    timeout=3600
)
def grpo_training(sft_model_path: str = "/models/merged_finetuned_qwen"):
    """
    Phase 2: GRPO (Group Relative Policy Optimization) training with custom reward function.
    
    Args:
        sft_model_path: Path to the SFT model from Phase 1
        
    Returns:
        str: Path to the final GRPO-trained model
    """
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from datasets import load_dataset
    from art import GRPOTrainer
    import os
    import sys
    import tempfile
    
    try:
        print("\n" + "="*80)
        print("🚀 PHASE 2: GRPO/ART REINFORCEMENT LEARNING")
        print("="*80 + "\n")
        
        # Write reward.py to a temporary file and import it
        print("📝 Setting up reward function...")
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(REWARD_CODE)
            reward_path = f.name
        
        sys.path.insert(0, os.path.dirname(reward_path))
        from reward import compute_reward
        print("✅ Reward function loaded\n")
        
        # Load tokenizer and model
        print(f"📥 Loading SFT model from: {sft_model_path}")
        tokenizer = AutoTokenizer.from_pretrained(sft_model_path)
        model = AutoModelForCausalLM.from_pretrained(
            sft_model_path,
            device_map="auto",
            torch_dtype=torch.float16
        )
        print("✅ Model loaded\n")
        
        # Load dataset for GRPO
        print("📥 Loading dataset for GRPO training...")
        raw_dataset = load_dataset("virattt/financial-qa-10K", split="train")
        raw_dataset = raw_dataset.shuffle(seed=42).select(range(1000))
        print(f"✅ Dataset loaded: {len(raw_dataset)} examples\n")

        # Prepare dataset for GRPO
        def prepare_grpo_dataset(example):
            messages = [
                {"role": "system", "content": "You are a financial expert. Provide a concise answer to the question based on the given context."},
                {"role": "user", "content": f"Context: {example['context']}\n\nQuestion: {example['question']}"},
            ]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            return {
                "prompt": prompt,
                "ground_truth": example["answer"]
            }

        print("🔄 Formatting dataset for GRPO...")
        grpo_dataset = raw_dataset.map(prepare_grpo_dataset)
        print("✅ Dataset formatted\n")

        # Create reward function
        def create_dataset_reward_fn(dataset):
            prompt_to_gt = {}
            for item in dataset:
                prompt_to_gt[item["prompt"]] = item["ground_truth"]
            
            def dataset_reward_fn(prompts, generations, **kwargs):
                rewards = []
                for prompt, generation in zip(prompts, generations):
                    ground_truth = prompt_to_gt.get(prompt, "")
                    reward = compute_reward(prompt, generation, ground_truth)
                    rewards.append(reward)
                return rewards
            
            return dataset_reward_fn

        print("🔧 Creating reward function wrapper...")
        reward_fn = create_dataset_reward_fn(grpo_dataset)
        print("✅ Reward function ready\n")

        # Initialize GRPO trainer
        print("🔧 Initializing GRPOTrainer...")
        rl_trainer = GRPOTrainer(
            model=model,
            tokenizer=tokenizer,
            reward_fn=reward_fn,
            rollout_batch_size=8,
            learning_rate=5e-6,
        )
        print("✅ GRPOTrainer initialized\n")

        # Train with GRPO
        print("🚀 Starting GRPO training...")
        print("-" * 80)
        rl_trainer.train(grpo_dataset)
        print("-" * 80)
        print("✅ GRPO training complete!\n")

        # Save final model
        print("💾 Saving GRPO-trained model...")
        os.makedirs("/models/qwen-4b-art", exist_ok=True)
        rl_trainer.save("/models/qwen-4b-art")
        
        volume.commit()
        
        print("\n" + "="*80)
        print("✅ PHASE 2 COMPLETE: GRPO/ART Model Saved")
        print("📍 Location: /models/qwen-4b-art")
        print("="*80 + "\n")
        
        return "/models/qwen-4b-art"
        
    except Exception as e:
        print(f"\n❌ ERROR in GRPO Training: {str(e)}")
        raise


In [ ]:
# ============================================================================
# PHASE 3: DSPy INFERENCE TESTING
# ============================================================================

@app.function(
    image=image,
    gpu="A10G",
    volumes={"/models": volume},
    timeout=600
)
def dspy_inference_test(model_path: str = "/models/qwen-4b-art"):
    """
    Phase 3: Test the trained model using DSPy with Chain-of-Thought reasoning.
    
    Args:
        model_path: Path to the trained model
        
    Returns:
        dict: Test results with examples and answers
    """
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import dspy
    
    try:
        print("\n" + "="*80)
        print("🚀 PHASE 3: DSPy INFERENCE TESTING")
        print("="*80 + "\n")
        
        # Load model and tokenizer
        print(f"📥 Loading model from: {model_path}")
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        # Set up DSPy with the trained model
        print("🔧 Configuring DSPy with trained model...")
        lm = dspy.HFModel(model=model_path, tokenizer=tokenizer, max_tokens=512, temperature=0.0)
        dspy.settings.configure(lm=lm)
        print("✅ DSPy configured\n")

        # Define Financial QA module with Chain-of-Thought
        class FinancialQA(dspy.Module):
            def __init__(self):
                super().__init__()
                self.generate_answer = dspy.ChainOfThought("context: str, question: str -> answer: str")

            def forward(self, context, question):
                prediction = self.generate_answer(context=context, question=question)
                return prediction.answer

        # Initialize QA module
        print("🔧 Initializing FinancialQA module...")
        qa = FinancialQA()
        print("✅ Module ready\n")

        # Test examples
        test_examples = [
            {
                "context": "Since our original focus on PC graphics, we have expanded to several large and important computationally intensive fields.",
                "question": "What area did NVIDIA initially focus on before expanding?"
            },
            {
                "context": "Total revenue for Q4 was $22.1 billion, up 265% from a year ago and up 22% from the previous quarter.",
                "question": "What was the year-over-year revenue growth percentage?"
            },
            {
                "context": "Our Data Center platform achieved record revenue of $18.4 billion, up 409% from a year earlier.",
                "question": "What was the Data Center revenue?"
            }
        ]

        results = []
        print("🧪 Running inference tests...")
        print("-" * 80)
        
        for i, example in enumerate(test_examples, 1):
            print(f"\n📝 Test Example {i}/{len(test_examples)}")
            print(f"Context: {example['context'][:100]}...")
            print(f"Question: {example['question']}")
            
            try:
                answer = qa(context=example["context"], question=example["question"])
                print(f"✅ Answer: {answer}\n")
                
                results.append({
                    "example": i,
                    "question": example["question"],
                    "answer": answer,
                    "status": "success"
                })
            except Exception as e:
                print(f"❌ Error: {str(e)}\n")
                results.append({
                    "example": i,
                    "question": example["question"],
                    "error": str(e),
                    "status": "failed"
                })
        
        print("-" * 80)
        
        # Summary
        successful = sum(1 for r in results if r["status"] == "success")
        print("\n" + "="*80)
        print(f"✅ PHASE 3 COMPLETE: DSPy Inference Testing")
        print(f"📊 Results: {successful}/{len(test_examples)} tests passed")
        print("="*80 + "\n")
        
        return {
            "model_path": model_path,
            "total_tests": len(test_examples),
            "successful": successful,
            "results": results
        }
        
    except Exception as e:
        print(f"\n❌ ERROR in DSPy Inference: {str(e)}")
        raise


In [ ]:
# ============================================================================
# HELPER FUNCTIONS: Download Models from Modal Volume
# ============================================================================

@app.function(
    image=image,
    volumes={"/models": volume},
    timeout=600
)
def download_models(model_name: str = "all", output_dir: str = "./downloaded_models"):
    """
    Download trained models from Modal volume to local directory.
    
    Args:
        model_name: Which model to download ("sft", "art", or "all")
        output_dir: Local directory to save models
        
    Returns:
        dict: Download status and paths
    """
    import os
    import shutil
    
    try:
        print("\n" + "="*80)
        print("📦 DOWNLOADING MODELS FROM MODAL VOLUME")
        print("="*80 + "\n")
        
        model_paths = {
            "sft": "/models/merged_finetuned_qwen",
            "art": "/models/qwen-4b-art"
        }
        
        results = {}
        
        if model_name == "all":
            models_to_download = model_paths.items()
        elif model_name in model_paths:
            models_to_download = [(model_name, model_paths[model_name])]
        else:
            raise ValueError(f"Invalid model_name: {model_name}. Choose 'sft', 'art', or 'all'")
        
        for name, src_path in models_to_download:
            print(f"📥 Downloading {name.upper()} model...")
            print(f"   Source: {src_path}")
            
            if os.path.exists(src_path):
                dest_path = os.path.join(output_dir, os.path.basename(src_path))
                os.makedirs(dest_path, exist_ok=True)
                
                # Copy files
                for item in os.listdir(src_path):
                    src_file = os.path.join(src_path, item)
                    dest_file = os.path.join(dest_path, item)
                    
                    if os.path.isfile(src_file):
                        shutil.copy2(src_file, dest_file)
                    elif os.path.isdir(src_file):
                        shutil.copytree(src_file, dest_file, dirs_exist_ok=True)
                
                print(f"   ✅ Downloaded to: {dest_path}")
                results[name] = {
                    "status": "success",
                    "path": dest_path
                }
            else:
                print(f"   ⚠️  Model not found: {src_path}")
                results[name] = {
                    "status": "not_found",
                    "path": None
                }
            print()
        
        print("="*80)
        print("✅ DOWNLOAD COMPLETE")
        print("="*80 + "\n")
        
        return results
        
    except Exception as e:
        print(f"\n❌ ERROR in Download: {str(e)}")
        raise


@app.function(
    image=image,
    volumes={"/models": volume},
    timeout=300
)
def list_models():
    """
    List all models available in the Modal volume.
    
    Returns:
        list: Available model paths with sizes
    """
    import os
    
    try:
        print("\n" + "="*80)
        print("📋 LISTING MODELS IN MODAL VOLUME")
        print("="*80 + "\n")
        
        models_dir = "/models"
        
        if not os.path.exists(models_dir):
            print("⚠️  Models directory not found")
            return []
        
        models = []
        for item in os.listdir(models_dir):
            item_path = os.path.join(models_dir, item)
            if os.path.isdir(item_path):
                # Calculate directory size
                total_size = 0
                for dirpath, dirnames, filenames in os.walk(item_path):
                    for filename in filenames:
                        filepath = os.path.join(dirpath, filename)
                        total_size += os.path.getsize(filepath)
                
                size_gb = total_size / (1024 ** 3)
                models.append({
                    "name": item,
                    "path": item_path,
                    "size_gb": round(size_gb, 2)
                })
                print(f"📦 {item}")
                print(f"   Path: {item_path}")
                print(f"   Size: {size_gb:.2f} GB\n")
        
        print("="*80)
        print(f"✅ Found {len(models)} model(s)")
        print("="*80 + "\n")
        
        return models
        
    except Exception as e:
        print(f"\n❌ ERROR in List Models: {str(e)}")
        raise


In [ ]:
# ============================================================================
# MAIN ORCHESTRATOR: Run Full Training Pipeline
# ============================================================================

@app.function(
    image=image,
    gpu="A10G",
    volumes={"/models": volume},
    timeout=7200
)
def train_full_pipeline(skip_sft: bool = False, skip_grpo: bool = False, skip_inference: bool = False):
    """
    Main orchestrator that runs the complete training pipeline.
    
    Args:
        skip_sft: Skip SFT training (use existing model)
        skip_grpo: Skip GRPO training (use SFT model only)
        skip_inference: Skip inference testing
        
    Returns:
        dict: Complete pipeline results
    """
    import time
    
    try:
        start_time = time.time()
        
        print("\n" + "="*80)
        print("🚀 STARTING COMPLETE TRAINING PIPELINE")
        print("="*80)
        print(f"Configuration:")
        print(f"  - Skip SFT: {skip_sft}")
        print(f"  - Skip GRPO: {skip_grpo}")
        print(f"  - Skip Inference: {skip_inference}")
        print("="*80 + "\n")
        
        results = {}
        
        # Phase 1: SFT Training
        if not skip_sft:
            print("🔄 Starting Phase 1: SFT Training...")
            sft_start = time.time()
            sft_path = sft_training.remote()
            sft_time = time.time() - sft_start
            results["sft"] = {
                "status": "completed",
                "path": sft_path,
                "time_minutes": round(sft_time / 60, 2)
            }
            print(f"✅ Phase 1 completed in {sft_time/60:.2f} minutes\n")
        else:
            sft_path = "/models/merged_finetuned_qwen"
            results["sft"] = {
                "status": "skipped",
                "path": sft_path
            }
            print("⏭️  Phase 1 skipped (using existing SFT model)\n")
        
        # Phase 2: GRPO Training
        if not skip_grpo:
            print("🔄 Starting Phase 2: GRPO Training...")
            grpo_start = time.time()
            grpo_path = grpo_training.remote(sft_path)
            grpo_time = time.time() - grpo_start
            results["grpo"] = {
                "status": "completed",
                "path": grpo_path,
                "time_minutes": round(grpo_time / 60, 2)
            }
            final_model_path = grpo_path
            print(f"✅ Phase 2 completed in {grpo_time/60:.2f} minutes\n")
        else:
            final_model_path = sft_path
            results["grpo"] = {
                "status": "skipped"
            }
            print("⏭️  Phase 2 skipped (using SFT model as final model)\n")
        
        # Phase 3: Inference Testing
        if not skip_inference:
            print("🔄 Starting Phase 3: DSPy Inference Testing...")
            inference_start = time.time()
            inference_results = dspy_inference_test.remote(final_model_path)
            inference_time = time.time() - inference_start
            results["inference"] = {
                "status": "completed",
                "results": inference_results,
                "time_minutes": round(inference_time / 60, 2)
            }
            print(f"✅ Phase 3 completed in {inference_time/60:.2f} minutes\n")
        else:
            results["inference"] = {
                "status": "skipped"
            }
            print("⏭️  Phase 3 skipped\n")
        
        # Final summary
        total_time = time.time() - start_time
        
        print("\n" + "="*80)
        print("🎉 COMPLETE TRAINING PIPELINE FINISHED!")
        print("="*80)
        print(f"⏱️  Total Time: {total_time/60:.2f} minutes ({total_time/3600:.2f} hours)")
        print(f"\n📊 Summary:")
        for phase, result in results.items():
            status = result.get("status", "unknown")
            if status == "completed":
                time_min = result.get("time_minutes", 0)
                print(f"  - {phase.upper()}: ✅ Completed ({time_min} min)")
            else:
                print(f"  - {phase.upper()}: ⏭️  Skipped")
        
        print(f"\n📍 Final Model: {final_model_path}")
        print("="*80 + "\n")
        
        results["total_time_minutes"] = round(total_time / 60, 2)
        results["final_model_path"] = final_model_path
        
        return results
        
    except Exception as e:
        print(f"\n❌ ERROR in Pipeline: {str(e)}")
        raise


In [ ]:
# ============================================================================
# NOTE: This cell has been refactored into modular functions above
# ============================================================================
# The old monolithic finetune_model() function has been split into:
#   - sft_training() - Cell 3
#   - grpo_training() - Cell 4  
#   - dspy_inference_test() - Cell 5
#   - train_full_pipeline() - Cell 7 (orchestrator)
#
# Use the individual functions or the orchestrator instead.
# ============================================================================

pass

In [ ]:
# ============================================================================
# LOCAL ENTRYPOINT: Choose what to run
# ============================================================================

@app.local_entrypoint()
def main():
    """
    Main entry point for Modal execution.
    
    Default: Runs the complete training pipeline (SFT + GRPO + Inference)
    
    Individual functions can be called with:
        modal run modal_finetune.ipynb::sft_training
        modal run modal_finetune.ipynb::grpo_training
        modal run modal_finetune.ipynb::dspy_inference_test
        modal run modal_finetune.ipynb::list_models
        modal run modal_finetune.ipynb::download_models
    """
    import sys
    
    print("\n" + "="*80)
    print("🚀 FinSure AI: Modal Training Pipeline")
    print("="*80 + "\n")
    
    # Run the complete pipeline by default
    print("Running complete training pipeline...")
    print("(To run individual phases, use: modal run modal_finetune.ipynb::<function_name>)\n")
    
    result = train_full_pipeline.remote()
    
    print("\n" + "="*80)
    print("✅ Pipeline Execution Complete!")
    print("="*80)
    print(f"\n📊 Results:")
    print(f"  - Total Time: {result.get('total_time_minutes', 0)} minutes")
    print(f"  - Final Model: {result.get('final_model_path', 'N/A')}")
    print(f"\n💡 To download models locally:")
    print(f"   modal run modal_finetune.ipynb::download_models")
    print("="*80 + "\n")
